# Core Pydantic Validation for LLMs
This notebook demonstrates basic validation, custom validators, handling 'messy' text using Regex, and modeling nested structures for deep data extraction using Pydantic v2.


In [4]:
%pip install pydantic[email]

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import json
import re
from pydantic import BaseModel, Field, EmailStr, field_validator, ValidationError
from typing import Optional, List

# 1. Basic Validation Schema with Custom Validator
class ContactInfo(BaseModel):
    name: str
    email: EmailStr
    phone: Optional[str] = None
    company: Optional[str] = None

    @field_validator('phone')
    @classmethod
    def validate_phone(cls, v):
        if v is None: 
            return v
        cleaned = ''.join(filter(str.isdigit, v))
        if len(cleaned) < 10:
            raise ValueError('Phone number must have at least 10 digits')
        return cleaned

# Validate perfect JSON data
print("--- 1. Testing Valid Basic Output ---")
llm_response_1 = '''
{
  "name": "Sarah Johnson",
  "email": "sarah.johnson@techcorp.com",
  "phone": "(555) 123-4567",
  "company": "TechCorp Industries"
}
'''
contact = ContactInfo(**json.loads(llm_response_1))
print(f"Successfully parsed contract for: {contact.name}")
print(contact.model_dump())

# 2. Parsing Messy LLM Responses (Extracting JSON from markdown/text)
class ProductReview(BaseModel):
    product_name: str
    rating: int
    review_text: str
    would_recommend: bool

    @field_validator('rating')
    @classmethod
    def validate_rating(cls, v):
        if not 1 <= v <= 5:
            raise ValueError('Rating must be an integer between 1 and 5')
        return v

def extract_json_from_llm_response(response: str) -> dict:
    """Extract JSON block from LLM responses containing markdown wrapping or extra text."""
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if json_match:
        return json.loads(json_match.group())
    raise ValueError("No JSON found in response")

def parse_review(llm_output: str) -> ProductReview:
    try:
        data = extract_json_from_llm_response(llm_output)
        return ProductReview(**data)
    except (json.JSONDecodeError, ValidationError, ValueError) as e:
        print(f"Extraction failed with error: {e}")
        raise

print("\n--- 2. Testing Messy Content Extraction ---")
messy_response = '''
Here's the product review you requested in JSON format:
{
  "product_name": "Wireless Headphones X100",
  "rating": 4,
  "review_text": "Great sound quality, comfortable for long use.",
  "would_recommend": true
}
Hope this helps! Feel free to ask for more.
'''
review = parse_review(messy_response)
print(f"Extracted Product: {review.product_name} | Rating: {review.rating}/5")

# 3. Handling Complex Nested Data Models
class Specification(BaseModel):
    key: str
    value: str

class Review(BaseModel):
    reviewer_name: str
    rating: int = Field(..., ge=1, le=5)
    comment: str
    verified_purchase: bool = False

class Product(BaseModel):
    id: str
    name: str
    price: float = Field(..., gt=0)
    category: str
    specifications: List[Specification]
    reviews: List[Review]
    average_rating: float = Field(..., ge=1, le=5)

    @field_validator('average_rating')
    @classmethod
    def check_average_matches_reviews(cls, v, info):
        reviews = info.data.get('reviews', [])
        if reviews:
            calculated_avg = sum(r.rating for r in reviews) / len(reviews)
            if abs(calculated_avg - v) > 0.1:
                raise ValueError(f'Average rating {v} does not match calculated average {calculated_avg:.2f}')
        return v

print("\n--- 3. Testing Complex Nested Structure ---")
nested_response = {
    "id": "PROD-2024-001",
    "name": "Smart Coffee Maker",
    "price": 129.99,
    "category": "Kitchen Appliances",
    "specifications": [
        {"key": "Capacity", "value": "12 cups"},
        {"key": "Power", "value": "1000W"}
    ],
    "reviews": [
        {"reviewer_name": "Alex M.", "rating": 5, "comment": "Makes excellent coffee!", "verified_purchase": True},
        {"reviewer_name": "Jordan P.", "rating": 4, "comment": "Good but a bit noisy", "verified_purchase": True}
    ],
    "average_rating": 4.5
}
product = Product(**nested_response)
print(f"Validated Nested Product Object: {product.name} (${product.price})")
print(f"Total verified reviews: {len(product.reviews)}")



--- 1. Testing Valid Basic Output ---
Successfully parsed contract for: Sarah Johnson
{'name': 'Sarah Johnson', 'email': 'sarah.johnson@techcorp.com', 'phone': '5551234567', 'company': 'TechCorp Industries'}

--- 2. Testing Messy Content Extraction ---
Extracted Product: Wireless Headphones X100 | Rating: 4/5

--- 3. Testing Complex Nested Structure ---
Validated Nested Product Object: Smart Coffee Maker ($129.99)
Total verified reviews: 2
